In [1]:
import pandas as pd
from matplotlib.pyplot import plot

In [2]:
path = "/home/viper/Desktop/Store Item Demand Forecasting Challenge/train.csv"

In [3]:
dataset = pd.read_csv(path)

In [4]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 913000 entries, 0 to 912999
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   date    913000 non-null  object
 1   store   913000 non-null  int64 
 2   item    913000 non-null  int64 
 3   sales   913000 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 27.9+ MB


In [5]:
dataset = dataset[dataset['store'] == 1]

In [6]:
dataset["date"] = pd.to_datetime(dataset["date"])
dataset.info()

<class 'pandas.core.frame.DataFrame'>
Index: 91300 entries, 0 to 896565
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    91300 non-null  datetime64[ns]
 1   store   91300 non-null  int64         
 2   item    91300 non-null  int64         
 3   sales   91300 non-null  int64         
dtypes: datetime64[ns](1), int64(3)
memory usage: 3.5 MB


In [7]:
dataset = dataset.drop(columns=["store"])

In [8]:
dataset["item"].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50])

In [9]:
# dataset = dataset.sort_values(["date", "item"]).reset_index(drop=True)

In [10]:
dataset.head()

,date,item,sales
0,2013-01-01,1,13
1,2013-01-02,1,11
2,2013-01-03,1,14
3,2013-01-04,1,13
4,2013-01-05,1,10


In [11]:
# 2. Grouped feature engineering
# dataset["sales_lag_1"] = dataset.groupby(["item"])["sales"].shift(1)
# dataset["sales_lag_7"] = dataset.groupby(["item"])["sales"].shift(7)
# dataset["sales_lag_30"] = dataset.groupby(["item"])["sales"].shift(30)

In [12]:
# 1. Sort chronologically per item
dataset =dataset.sort_values(['item', 'date']).reset_index(drop=True)

# 2. Calculate lags strictly within each item group
dataset['sales_lag_1'] =dataset.groupby('item')['sales'].shift(1)
dataset['sales_lag_7'] =dataset.groupby('item')['sales'].shift(7)

In [13]:
dataset.tail(50)

,date,item,sales,sales_lag_1,sales_lag_7
91250,2017-11-12,50,84,89.0,83.0
91251,2017-11-13,50,57,84.0,56.0
91252,2017-11-14,50,59,57.0,52.0
91253,2017-11-15,50,67,59.0,58.0
91254,2017-11-16,50,81,67.0,69.0
91255,2017-11-17,50,70,81.0,68.0
91256,2017-11-18,50,80,70.0,89.0
91257,2017-11-19,50,84,80.0,84.0
91258,2017-11-20,50,47,84.0,57.0
91259,2017-11-21,50,58,47.0,59.0


In [14]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91300 entries, 0 to 91299
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         91300 non-null  datetime64[ns]
 1   item         91300 non-null  int64         
 2   sales        91300 non-null  int64         
 3   sales_lag_1  91250 non-null  float64       
 4   sales_lag_7  90950 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(2)
memory usage: 3.5 MB


In [15]:
dataset.drop(columns=["store"])

KeyError: "['store'] not found in axis"

In [ ]:
dataset["item"] = dataset["item"].astype("category")


In [ ]:
dataset.drop(columns=["store"])

,date,item,sales,sales_lag_1,sales_lag_7,sales_lag_30
0,2013-01-01,1,13,NaN,NaN,NaN
1,2013-01-01,1,12,NaN,NaN,NaN
2,2013-01-01,1,19,NaN,NaN,NaN
3,2013-01-01,1,10,NaN,NaN,NaN
4,2013-01-01,1,11,NaN,NaN,NaN
...,...,...,...,...,...,...
912995,2017-12-31,50,45,41.0,50.0,46.0
912996,2017-12-31,50,46,44.0,34.0,40.0
912997,2017-12-31,50,76,70.0,81.0,88.0
912998,2017-12-31,50,65,62.0,66.0,66.0


In [ ]:
df_clean = dataset.dropna()

In [ ]:
df_clean

,date,store,item,sales,sales_lag_1,sales_lag_7,sales_lag_30
15000,2013-01-31,1,1,13,9.0,8.0,13.0
15001,2013-01-31,2,1,16,10.0,14.0,12.0
15002,2013-01-31,3,1,17,16.0,12.0,19.0
15003,2013-01-31,4,1,8,14.0,12.0,10.0
15004,2013-01-31,5,1,6,8.0,8.0,11.0
...,...,...,...,...,...,...,...
912995,2017-12-31,6,50,45,41.0,50.0,46.0
912996,2017-12-31,7,50,46,44.0,34.0,40.0
912997,2017-12-31,8,50,76,70.0,81.0,88.0
912998,2017-12-31,9,50,65,62.0,66.0,66.0


In [ ]:
split_date = "2017-10-01"
train = df_clean[df_clean["date"] < split_date]
test = df_clean[df_clean["date"] >= split_date]

features = ["store", "item", "sales_lag_1", "sales_lag_7"]
X_train, y_train = train[features], train["sales"]
X_test, y_test = test[features], test["sales"]

In [ ]:
import lightgbm as lgb

# 5. Train single global model
model = lgb.LGBMRegressor(n_estimators=300, random_state=42)
model.fit(X_train, y_train, categorical_feature=["store", "item"])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013077 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 440
[LightGBM] [Info] Number of data points in the train set: 852000, number of used features: 4
[LightGBM] [Info] Start training from score 52.522494


,n_estimators,300
,random_state,42
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001


In [ ]:
model.predict()

TypeError: LGBMModel.predict() missing 1 required positional argument: 'X'